## Anreillas 2026 Bulk RNA seq data processing and AnnData obj creation

In [5]:
import os
import pandas as pd

# Point this directly to the folder that already exists
extract_dir = "/Users/marikaclark/dark_matter_atx/data/Anerillas_2026/GSE284666_RAW"

# Step 2: Loop through the .gz files and merge them side-by-side
expression_data = {}

# Get a sorted list of all the .gz files in the folder
gz_files = sorted([f for f in os.listdir(extract_dir) if f.endswith('.gz')])

if len(gz_files) == 0:
    raise FileNotFoundError(f"No .gz files found in {extract_dir}. Please double check the folder contents.")

print(f"Found {len(gz_files)} compressed sample files. Starting merge...")

for file_name in gz_files:
    # Extract a clean sample ID from the filename (e.g., "GSM8690061")
    sample_id = file_name.split('_')[0] 
    file_path = os.path.join(extract_dir, file_name)
    
    # Read the individual file (pandas handles .gz decompression automatically)
    # Note: Using header=None and names assumes a standard 2-column layout (Gene, Count)
    df = pd.read_csv(file_path, sep="\t", header=None, names=["Gene_ID", "Count"])
    
    # Set the Gene_ID as index and save the counts under the sample's ID
    df.set_index("Gene_ID", inplace=True)
    expression_data[sample_id] = df["Count"]

# Step 3: Combine all individual series into a single matrix DataFrame
merged_expr_df = pd.DataFrame(expression_data)

print("\n--- Final Consolidated Matrix ---")
print("Matrix Shape (Genes x Samples):", merged_expr_df.shape)
print(merged_expr_df.head())

# Save your clean matrix so you don't have to rebuild it next time!
output_path = os.path.join(os.path.dirname(extract_dir), "consolidated_expression_matrix.csv")
merged_expr_df.to_csv(output_path)
print(f"Saved consolidated matrix to: {output_path}")

Found 190 compressed sample files. Starting merge...

--- Final Consolidated Matrix ---
Matrix Shape (Genes x Samples): (60606, 190)
                            GSM8690061             GSM8690062  \
Gene_ID                                                         
Gene_stable_ID   S_65_BJ_P_1_geneCOUNT  S_66_BJ_P_2_geneCOUNT   
ENSG00000000003                   6291                   5953   
ENSG00000000005                      0                      0   
ENSG00000000419                   4733                   4082   
ENSG00000000457                   2008                   1319   

                            GSM8690063             GSM8690064  \
Gene_ID                                                         
Gene_stable_ID   S_67_BJ_P_3_geneCOUNT  S_68_BJ_P_4_geneCOUNT   
ENSG00000000003                   5043                   6285   
ENSG00000000005                      0                      0   
ENSG00000000419                   3781                   4331   
ENSG00000000457      